# ETTh эксперименты: LSTM + Optuna

В этом ноутбуке запускаются эксперименты **LSTM + Optuna** на датасетах `ETTh1` и `ETTh2`.

Основные шаги:

1. Настройка окружения и конфигурации экспериментов.
2. Запуск поиска гиперпараметров LSTM с помощью Optuna для разных значений `MAX_ROWS`.
3. Повторное обучение лучшей LSTM-модели для визуализации.
4. Построение линейных графиков фактических и предсказанных значений временного ряда отдельно для train и valid частей выборки.
5. Формирование сводной таблицы метрик по всем запускам.

In [ ]:
from __future__ import annotations

import logging
import os
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Dict
from typing import List
from typing import Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv

CURRENT_DIR: Path = Path.cwd().resolve()
REPO_ROOT_CANDIDATES: List[Path] = [
    CURRENT_DIR,
    CURRENT_DIR.parent,
]

REPO_ROOT: Path | None = None
for candidate in REPO_ROOT_CANDIDATES:
    if (candidate / 'src' / 'edlm_search').is_dir():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise RuntimeError(
            f'Cannot locate project root with "src/edlm_search" directory from "{CURRENT_DIR}".'
    )

SRC_DIR: Path = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from edlm_search.experiments import ExperimentResult, run_lstm_optuna_etth_experiment
from edlm_search.experiments.datasets import load_ett_csv_dataset
from edlm_search.baselines.baseline_optuna import (
    SimpleLSTMForecaster,
    create_dataloaders_for_etth,
    _build_autoregressive_predictions,
)
from edlm_search.devices import get_torch_device

In [ ]:
ENV_PATH: Path = REPO_ROOT / '.env'
if ENV_PATH.is_file():
    load_dotenv(dotenv_path=ENV_PATH)

ETT_DATA_DIR: Path = SRC_DIR / 'ETDataset' / 'ETT-small'
ETTH1_PATH: Path = ETT_DATA_DIR / 'ETTh1.csv'
ETTH2_PATH: Path = ETT_DATA_DIR / 'ETTh2.csv'

DATASET_NAMES: List[str] = ['ETTh1', 'ETTh2']

MAX_ROWS_VALUES: List[int] = [5000, 10000, 20000]
TRAIN_RATIO: float = float(os.getenv('TRAIN_RATIO', '0.8'))
TARGET_COLUMN: str = os.getenv('TARGET_COLUMN', 'OT')

LSTM_N_TRIALS: int = int(os.getenv('N_TRIALS', '20'))
LSTM_MODEL_NAME: str = os.getenv('LSTM_MODEL_NAME', 'lstm-optuna')
ARTIFACTS_DIR: Path = (REPO_ROOT / os.getenv('ARTIFACTS_DIR', 'artifacts').strip()).resolve()
DEVICE_TYPE: str = os.getenv('DEVICE_TYPE', 'auto')

LOG_LEVEL: int = logging.INFO
logging.basicConfig(
        level=LOG_LEVEL,
        format='%(asctime)s - %(levelname)s - %(name)s - %(message)s',
)

LOGGER = logging.getLogger('etth_lstm_optuna_experiments')

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

LOGGER.info(f'Repository root resolved to "{REPO_ROOT}".')
LOGGER.info(f'ETT data directory resolved to "{ETT_DATA_DIR}".')
LOGGER.info(f'Artifacts directory resolved to "{ARTIFACTS_DIR}".')
LOGGER.info(f'LSTM Optuna experiments will use device_type="{DEVICE_TYPE}".')

## Конфигурация и структуры данных

В этом разделе определяются:

- описания датасетов;
- конфигурация экспериментов LSTM + Optuna;
- служебные структуры для идентификации запусков.

In [ ]:
@dataclass(frozen=True)
class DatasetConfig:
    """Dataset configuration for ETTh experiments."""

    name: str
    csv_path: Path
    train_ratio: float


@dataclass(frozen=True)
class LSTMOptunaExperimentConfig:
    """Configuration for LSTM + Optuna experiments."""

    n_trials: int
    target_column: str
    model_name: str
    artifacts_dir: Path
    device_type: str
    max_rows_values: List[int]


@dataclass(frozen=True)
class LSTMRunKey:
    """Identifier of a single experiment run."""

    dataset_name: str
    max_rows: int

## Класс для обучения и получения предсказаний LSTM

Следующий класс отвечает за:

- обучение LSTM-модели с заданными гиперпараметрами;
- получение предсказаний на train и valid частях выборки;
- подготовку данных для построения графиков.

In [ ]:
class LSTMVisualizationTrainer:
    """Train LSTM model with given hyperparameters and return train/validation predictions."""

    def __init__(self, device: torch.device, target_column: str) -> None:
        if target_column == '':
            raise ValueError('target_column must not be empty.')
        self._device = device
        self._target_column = target_column

    def train_and_predict(
            self,
            train_df: pd.DataFrame,
            valid_df: pd.DataFrame,
            hyperparams: Dict[str, float],
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """
        Train LSTM model and return predictions for train and validation sets.

        Returns
        -------
        tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]
            Flattened (y_true_train, y_pred_train, y_true_valid, y_pred_valid).
        """
        self._validate_input_frames(train_df, valid_df)
        params = self._build_hyperparams(hyperparams)

        feature_columns = self._build_feature_columns(train_df)
        self._validate_feature_columns(feature_columns, valid_df)

        train_loader = self._build_train_loader(
                train_df=train_df,
                valid_df=valid_df,
                params=params,
                feature_columns=feature_columns,
        )

        model = self._build_model(
                input_size=len(feature_columns),
                params=params,
        )

        learning_rate = float(params['learning_rate'])

        self._train_model(
                model=model,
                train_loader=train_loader,
                num_epochs=int(params['num_epochs']),
                learning_rate=learning_rate,
        )

        full_df = self._build_full_dataframe(train_df=train_df, valid_df=valid_df)
        y_true_full, y_pred_full = self._build_autoregressive_predictions_for_full(
                model=model,
                full_df=full_df,
                feature_columns=feature_columns,
                seq_len=int(params['seq_len']),
        )

        train_len = len(train_df)
        seq_len = int(params['seq_len'])
        if train_len <= seq_len:
            raise ValueError(
                    'Training DataFrame must contain more rows than seq_len for visualization.'
            )

        train_pred_count = train_len - seq_len
        if train_pred_count < 0:
            raise ValueError('Invalid train prediction length computed for visualization.')

        y_true_train = y_true_full[:train_pred_count]
        y_pred_train = y_pred_full[:train_pred_count]
        y_true_valid = y_true_full[train_pred_count:]
        y_pred_valid = y_pred_full[train_pred_count:]

        return y_true_train, y_pred_train, y_true_valid, y_pred_valid

    def _validate_input_frames(
            self,
            train_df: pd.DataFrame,
            valid_df: pd.DataFrame,
    ) -> None:
        """Validate that input data frames have the required target column and non-zero length."""
        for frame, frame_name in ((train_df, 'train_df'), (valid_df, 'valid_df')):
            if self._target_column not in frame.columns:
                raise ValueError(
                        f'DataFrame "{frame_name}" must contain target column "{self._target_column}".'
                )
        if len(train_df) == 0:
            raise ValueError('train_df must not be empty.')
        if len(valid_df) == 0:
            raise ValueError('valid_df must not be empty.')

    @staticmethod
    def _build_hyperparams(
            hyperparams: Dict[str, float],
    ) -> Dict[str, int | float]:
        """Validate and normalize hyperparameters dictionary."""
        required_keys = [
            'seq_len',
            'pred_len',
            'hidden_size',
            'num_layers',
            'learning_rate',
            'batch_size',
            'num_epochs',
        ]
        normalized: Dict[str, int | float] = {}
        for key in required_keys:
            if key not in hyperparams:
                raise ValueError(f'Key "{key}" is missing in hyperparameters.')
            value = hyperparams[key]
            if key == 'learning_rate':
                value_float = float(value)
                if value_float <= 0.0:
                    raise ValueError(f'Hyperparameter "{key}" must be positive.')
                normalized[key] = value_float
            else:
                value_int = int(value)
                if value_int <= 0:
                    raise ValueError(f'Hyperparameter "{key}" must be a positive integer.')
                normalized[key] = value_int
        return normalized

    @staticmethod
    def _build_feature_columns(train_df: pd.DataFrame) -> List[str]:
        """Build list of feature columns for LSTM training and inference."""
        columns = [c for c in train_df.columns if c != 'date']
        if not columns:
            raise ValueError('Training DataFrame must contain at least one feature column.')
        return columns

    @staticmethod
    def _validate_feature_columns(feature_columns: List[str], valid_df: pd.DataFrame) -> None:
        """Validate that validation DataFrame contains all required feature columns."""
        missing = [c for c in feature_columns if c not in valid_df.columns]
        if missing:
            raise ValueError(
                    f'Validation DataFrame is missing feature columns: {missing}.'
            )

    def _build_train_loader(
            self,
            train_df: pd.DataFrame,
            valid_df: pd.DataFrame,
            params: Dict[str, int | float],
            feature_columns: List[str],
    ) -> torch.utils.data.DataLoader:
        """Build DataLoader for training using shared helper for ETTh datasets."""
        seq_len = int(params['seq_len'])
        pred_len = int(params['pred_len'])
        batch_size = int(params['batch_size'])

        train_loader, _, _ = create_dataloaders_for_etth(
                train_df=train_df,
                valid_df=valid_df,
                seq_len=seq_len,
                pred_len=pred_len,
                batch_size=batch_size,
                target_column=self._target_column,
        )
        return train_loader

    def _build_model(
            self,
            input_size: int,
            params: Dict[str, int | float],
    ) -> SimpleLSTMForecaster:
        """Create LSTM model instance for visualization training."""
        hidden_size = int(params['hidden_size'])
        num_layers = int(params['num_layers'])
        pred_len = int(params['pred_len'])

        model = SimpleLSTMForecaster(
                input_size=input_size,
                hidden_size=hidden_size,
                num_layers=num_layers,
                pred_len=pred_len,
        ).to(self._device)
        return model

    def _train_model(
            self,
            model: SimpleLSTMForecaster,
            train_loader: torch.utils.data.DataLoader,
            num_epochs: int,
            learning_rate: float,
    ) -> None:
        """Train LSTM model on provided training loader."""
        if num_epochs <= 0:
            raise ValueError('num_epochs must be positive for visualization training.')
        if learning_rate <= 0.0:
            raise ValueError('learning_rate must be positive for visualization training.')

        criterion = torch.nn.MSELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

        for epoch_index in range(num_epochs):
            LOGGER.info(
                    f'LSTM visualization training started: epoch={epoch_index + 1}/{num_epochs}, '
                    f'learning_rate={learning_rate}.'
            )
            model.train()
            epoch_loss_sum = 0.0
            batch_count = 0
            for batch_x, batch_y in train_loader:
                batch_x_device = batch_x.to(self._device)
                batch_y_device = batch_y.to(self._device)
                optimizer.zero_grad()
                outputs = model(batch_x_device)
                loss = criterion(outputs, batch_y_device)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                epoch_loss_sum += float(loss.item())
                batch_count += 1
            if batch_count > 0:
                avg_loss = epoch_loss_sum / float(batch_count)
                LOGGER.info(
                        f'LSTM visualization training finished: epoch={epoch_index + 1}/{num_epochs}, '
                        f'avg_loss={avg_loss:.6f}.'
                )

    @staticmethod
    def _build_full_dataframe(
            train_df: pd.DataFrame,
            valid_df: pd.DataFrame,
    ) -> pd.DataFrame:
        """Concatenate train and validation DataFrames for full-sequence forecasting."""
        full_df = pd.concat([train_df, valid_df], axis=0, ignore_index=True)
        if len(full_df) == 0:
            raise ValueError('Full DataFrame for forecasting must not be empty.')
        return full_df

    def _build_autoregressive_predictions_for_full(
            self,
            model: SimpleLSTMForecaster,
            full_df: pd.DataFrame,
            feature_columns: List[str],
            seq_len: int,
    ) -> Tuple[np.ndarray, np.ndarray]:
        """Build autoregressive one-step-ahead predictions for the full concatenated DataFrame."""
        if seq_len <= 0:
            raise ValueError('seq_len must be a positive integer for autoregressive prediction.')

        if self._target_column not in full_df.columns:
            raise ValueError(
                    f'Full DataFrame must contain target column "{self._target_column}" '
                    f'for autoregressive prediction.'
            )

        if self._target_column not in feature_columns:
            raise ValueError(
                    f'Target column "{self._target_column}" must be present in feature_columns '
                    f'for autoregressive prediction.'
            )

        y_true, y_pred = _build_autoregressive_predictions(
                model=model,
                data_frame=full_df,
                feature_columns=feature_columns,
                target_column=self._target_column,
                seq_len=seq_len,
                device=self._device,
                history_frame=None,
        )
        return y_true, y_pred

## Класс запуска экспериментов LSTM + Optuna

Этот класс:

- запускает эксперименты для всех датасетов и значений `MAX_ROWS`;
- формирует сводную таблицу метрик;
- загружает гиперпараметры из диагностического JSON для последующей визуализации.

In [ ]:
import json


class LSTMOptunaExperimentRunner:
    """Manage LSTM + Optuna experiments for multiple datasets and max_rows values."""

    def __init__(
            self,
            dataset_configs: List[DatasetConfig],
            experiment_config: LSTMOptunaExperimentConfig,
    ) -> None:
        if not dataset_configs:
            raise ValueError('dataset_configs must not be empty.')
        self._dataset_configs = list(dataset_configs)
        self._experiment_config = experiment_config

    def run_all(self) -> Dict[LSTMRunKey, ExperimentResult]:
        """Run experiments for all datasets and all max_rows values."""
        results: Dict[LSTMRunKey, ExperimentResult] = {}
        for dataset_config in self._dataset_configs:
            for max_rows in self._experiment_config.max_rows_values:
                run_key = LSTMRunKey(dataset_name=dataset_config.name, max_rows=max_rows)
                LOGGER.info(
                        f'Starting LSTM+Optuna run for dataset="{run_key.dataset_name}", '
                        f'max_rows={run_key.max_rows}.'
                )
                result = self._run_single(dataset_config, max_rows)
                results[run_key] = result
                mse_value = float(result.metrics.get('mse', float('nan')))
                LOGGER.info(
                        f'Finished LSTM+Optuna run for dataset="{run_key.dataset_name}", '
                        f'max_rows={run_key.max_rows}, mse={mse_value}.'
                )
        return results

    def _run_single(
            self,
            dataset_config: DatasetConfig,
            max_rows: int,
    ) -> ExperimentResult:
        """Run a single LSTM + Optuna experiment for a dataset and max_rows."""
        if max_rows <= 0:
            raise ValueError('max_rows must be a positive integer.')

        if not dataset_config.csv_path.is_file():
            raise FileNotFoundError(
                    f'Dataset "{dataset_config.name}" CSV not found at "{dataset_config.csv_path}".'
            )

        result = run_lstm_optuna_etth_experiment(
                dataset_name=dataset_config.name,
                csv_path=str(dataset_config.csv_path),
                max_rows=max_rows,
                train_ratio=dataset_config.train_ratio,
                n_trials=self._experiment_config.n_trials,
                target_column=self._experiment_config.target_column,
                model_name=self._experiment_config.model_name,
                artifacts_dir=str(self._experiment_config.artifacts_dir),
                device_type=self._experiment_config.device_type,
        )
        return result

    @staticmethod
    def build_metrics_dataframe(
            results: Dict[LSTMRunKey, ExperimentResult],
            primary_metric: str,
    ) -> pd.DataFrame:
        """Convert experiment results into a flat metrics DataFrame."""
        rows: List[Dict[str, float | int | str]] = []
        for run_key, result in results.items():
            metrics = result.metrics
            row: Dict[str, float | int | str] = {
                'dataset': run_key.dataset_name,
                'max_rows': run_key.max_rows,
            }
            for metric_name, metric_value in metrics.items():
                row[metric_name] = float(metric_value)
            if primary_metric not in row:
                row[primary_metric] = float('nan')
            rows.append(row)
        if not rows:
            return pd.DataFrame()
        df = pd.DataFrame(rows)
        df.sort_values(by=['dataset', 'max_rows'], inplace=True)
        df.reset_index(drop=True, inplace=True)
        return df

    @staticmethod
    def load_hyperparams_from_diagnostics(
            diagnostics_path: Path,
    ) -> Dict[str, float]:
        """Load best LSTM hyperparameters from diagnostics JSON file."""
        if not diagnostics_path.is_file():
            raise FileNotFoundError(
                    f'Diagnostics JSON file not found at "{diagnostics_path}".'
            )
        with diagnostics_path.open('r', encoding='utf-8') as file:
            payload = json.load(file)
        hyperparams_section = payload.get('hyperparams')
        if not isinstance(hyperparams_section, dict):
            raise ValueError(
                    f'Diagnostics JSON at "{diagnostics_path}" must contain "hyperparams" object.'
            )
        hyperparams: Dict[str, float] = {}
        for key, value in hyperparams_section.items():
            hyperparams[key] = float(value)
        return hyperparams

    @staticmethod
    def split_dataset_for_visualization(
            csv_path: Path,
            max_rows: int,
            train_ratio: float,
    ) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """Load and split ETTh dataset for visualization purposes."""
        train_df, valid_df = load_ett_csv_dataset(
                csv_path=str(csv_path),
                max_rows=max_rows,
                train_ratio=train_ratio,
        )
        return train_df, valid_df

## Вспомогательная функция визуализации

Эта функция строит линейный график фактического и предсказанного временного ряда.

In [ ]:
def plot_time_series(
        x_values: np.ndarray,
        y_true: np.ndarray,
        y_pred: np.ndarray,
        title: str,
        max_points: int,
) -> None:
    """
    Plot line chart for ground truth and predictions.

    Parameters
    ----------
    x_values : np.ndarray
        Values for the x-axis (typically timestamps or time indices).
    y_true : np.ndarray
        Ground truth values.
    y_pred : np.ndarray
        Predicted values.
    title : str
        Title of the plot.
    max_points : int
        Maximum number of points to display (full series is plotted).
    """
    if max_points <= 0:
        raise ValueError('max_points must be positive.')
    if x_values.ndim != 1:
        raise ValueError('x_values must be one-dimensional.')
    if y_true.ndim != 1:
        raise ValueError('y_true must be one-dimensional.')
    if y_pred.ndim != 1:
        raise ValueError('y_pred must be one-dimensional.')
    if x_values.shape[0] == 0:
        raise ValueError('x_values must be non-empty.')
    if y_true.shape[0] == 0:
        raise ValueError('y_true must be non-empty.')
    if y_pred.shape[0] == 0:
        raise ValueError('y_pred must be non-empty.')
    if x_values.shape[0] != y_true.shape[0] or x_values.shape[0] != y_pred.shape[0]:
        raise ValueError('x_values, y_true and y_pred must have the same length for plotting.')

    x_plot = x_values
    y_true_plot = y_true
    y_pred_plot = y_pred

    plt.figure(figsize=(16, 4))
    plt.plot(x_plot, y_true_plot, label='y_true')
    plt.plot(x_plot, y_pred_plot, label='y_pred')
    plt.xlabel('time')
    plt.ylabel('value')
    plt.title(title)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

## Запуск экспериментов LSTM + Optuna

В этом разделе:

1. Создаются конфигурации датасетов и эксперимента.
2. Запускаются все эксперименты LSTM + Optuna для комбинаций датасет × `MAX_ROWS`.
3. Формируется таблица метрик по всем запускам.

In [ ]:
dataset_configs: List[DatasetConfig] = [
    DatasetConfig(name='ETTh1', csv_path=ETTH1_PATH, train_ratio=TRAIN_RATIO),
    # DatasetConfig(name='ETTh2', csv_path=ETTH2_PATH, train_ratio=TRAIN_RATIO),
]

lstm_experiment_config = LSTMOptunaExperimentConfig(
        n_trials=LSTM_N_TRIALS,
        target_column=TARGET_COLUMN,
        model_name=LSTM_MODEL_NAME,
        artifacts_dir=ARTIFACTS_DIR,
        device_type=DEVICE_TYPE,
        max_rows_values=MAX_ROWS_VALUES,
)

runner = LSTMOptunaExperimentRunner(
        dataset_configs=dataset_configs,
        experiment_config=lstm_experiment_config,
)

results_by_run_key: Dict[LSTMRunKey, ExperimentResult] = runner.run_all()

primary_metric_name: str = 'mse'
metrics_df = LSTMOptunaExperimentRunner.build_metrics_dataframe(
        results=results_by_run_key,
        primary_metric=primary_metric_name,
)
metrics_df

## Визуализация предсказаний LSTM

На этом шаге:

1. Для каждого запуска загружаются лучшие гиперпараметры из диагностического JSON.
2. Заново загружаются данные и выполняется обучение LSTM-модели для визуализации.
3. Строятся два графика для каждой комбинации датасет × `MAX_ROWS`:
   - фактические и предсказанные значения для train;
   - фактические и предсказанные значения для valid.

In [ ]:
device = get_torch_device(DEVICE_TYPE)
visualization_trainer = LSTMVisualizationTrainer(
        device=device,
        target_column=TARGET_COLUMN,
)

dataset_config_by_name: Dict[str, DatasetConfig] = {
    cfg.name: cfg for cfg in dataset_configs
}

for run_key, result in results_by_run_key.items():
    diagnostics_path_value = result.extra_info.get('diagnostics_json_path')
    if not isinstance(diagnostics_path_value, str):
        LOGGER.info(
                f'Visualization skipped for dataset="{run_key.dataset_name}", '
                f'max_rows={run_key.max_rows}: diagnostics_json_path is missing.'
        )
        continue

    diagnostics_path = Path(diagnostics_path_value)
    try:
        hyperparams = LSTMOptunaExperimentRunner.load_hyperparams_from_diagnostics(
                diagnostics_path=diagnostics_path,
        )
    except Exception as exc:
        LOGGER.info(
                f'Visualization skipped for dataset="{run_key.dataset_name}", '
                f'max_rows={run_key.max_rows}: failed to load hyperparameters: {exc}.'
        )
        continue

    dataset_cfg = dataset_config_by_name.get(run_key.dataset_name)
    if dataset_cfg is None:
        LOGGER.info(
                f'Visualization skipped for dataset="{run_key.dataset_name}", '
                f'max_rows={run_key.max_rows}: dataset config not found.'
        )
        continue

    train_df, valid_df = LSTMOptunaExperimentRunner.split_dataset_for_visualization(
            csv_path=dataset_cfg.csv_path,
            max_rows=run_key.max_rows,
            train_ratio=dataset_cfg.train_ratio,
    )

    LOGGER.info(
            f'Starting visualization training for dataset="{run_key.dataset_name}", '
            f'max_rows={run_key.max_rows}.'
    )
    train_true, train_pred, valid_true, valid_pred = visualization_trainer.train_and_predict(
            train_df=train_df,
            valid_df=valid_df,
            hyperparams=hyperparams,
    )
    LOGGER.info(
            f'Visualization training finished for dataset="{run_key.dataset_name}", '
            f'max_rows={run_key.max_rows}.'
    )

    seq_len_value = int(hyperparams['seq_len'])

    if 'date' in train_df.columns:
        x_train_full = pd.to_datetime(train_df['date']).to_numpy()
        LOGGER.info(
                f'Using "date" column as x-axis for LSTM train plot '
                f'(dataset="{run_key.dataset_name}", max_rows={run_key.max_rows}).'
        )
    else:
        x_train_full = np.arange(len(train_df), dtype=int)
        LOGGER.info(
                f'Column "date" is missing in train_df, using positional index as x-axis '
                f'for LSTM train plot (dataset="{run_key.dataset_name}", max_rows={run_key.max_rows}).'
        )

    if 'date' in valid_df.columns:
        x_valid_full = pd.to_datetime(valid_df['date']).to_numpy()
        LOGGER.info(
                f'Using "date" column as x-axis for LSTM validation plot '
                f'(dataset="{run_key.dataset_name}", max_rows={run_key.max_rows}).'
        )
    else:
        x_valid_full = np.arange(len(valid_df), dtype=int)
        LOGGER.info(
                f'Column "date" is missing in valid_df, using positional index as x-axis '
                f'for LSTM validation plot (dataset="{run_key.dataset_name}", max_rows={run_key.max_rows}).'
        )

    train_len = len(train_df)
    if train_len <= seq_len_value:
        raise RuntimeError(
                f'Train length is not greater than seq_len for dataset="{run_key.dataset_name}", '
                f'max_rows={run_key.max_rows}.'
        )

    expected_train_points = train_len - seq_len_value
    expected_valid_points = len(valid_df)

    if train_true.shape[0] != expected_train_points:
        raise RuntimeError(
                f'Train predictions length mismatch for dataset="{run_key.dataset_name}", '
                f'max_rows={run_key.max_rows}: '
                f'expected {expected_train_points}, got {train_true.shape[0]}.'
        )
    if valid_true.shape[0] != expected_valid_points:
        raise RuntimeError(
                f'Validation predictions length mismatch for dataset="{run_key.dataset_name}", '
                f'max_rows={run_key.max_rows}: '
                f'expected {expected_valid_points}, got {valid_true.shape[0]}.'
        )

    x_train_values = x_train_full[seq_len_value:]
    x_valid_values = x_valid_full

    plot_time_series(
            x_values=x_train_values,
            y_true=train_true,
            y_pred=train_pred,
            title=(
                f'LSTM+Optuna train predictions '
                f'(dataset={run_key.dataset_name}, max_rows={run_key.max_rows})'
            ),
            max_points=train_true.shape[0],
    )
    plot_time_series(
            x_values=x_valid_values,
            y_true=valid_true,
            y_pred=valid_pred,
            title=(
                f'LSTM+Optuna validation predictions '
                f'(dataset={run_key.dataset_name}, max_rows={run_key.max_rows})'
            ),
            max_points=valid_true.shape[0],
    )